# GRAM — Colab training run (step 8)
Runtime → Change runtime type → **GPU**. Everything persistent (code zip, data cache, runs) lives in `MyDrive/gram/`.

1. Upload `gram_code.zip` (made locally with `git archive -o dist/gram_code.zip HEAD`) to `MyDrive/gram/`.
2. Run all cells. If the session dies, re-run all cells: training resumes from the last checkpoint on Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/gram'

In [ ]:
# Code: unzip the archive from Drive (or replace with `!git clone <your repo> /content/gram`).
!rm -rf /content/gram && mkdir -p /content/gram && unzip -q {WORK}/gram_code.zip -d /content/gram
%cd /content/gram
!pip -q install "tokenizers>=0.15"
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# Data: build once (~15-20 min, downloads ~830 MB from Hugging Face), then reuse the cached tar from Drive
# so every run trains on byte-identical data.
import os
TAR = f'{WORK}/data_processed.tar'
if os.path.exists(TAR):
    !mkdir -p data && tar -xf {TAR} -C data
else:
    !python -m gram.data --out data/processed --raw /content/raw
    !tar -cf {TAR} -C data processed
!python -c "import json; m=json.load(open('data/processed/meta.json')); print('tokenizer', m['tokenizer_sha256'][:16], {d: sum(s['tokens'] for s in v.values()) for d, v in m['counts'].items()})"

In [ ]:
# Train GRAM (p_as=0.3, p_cr=0.5). 10k steps x 64 x 256 = 164M tokens. Resumes automatically.
RUN = 'gram_s0'
OUT = f'{WORK}/runs/{RUN}'
CKPT = f'{OUT}/ckpt.pt'
if os.path.exists(CKPT):
    args = f'--resume {CKPT}'
else:
    args = f'--data-dir data/processed --out {OUT} --steps 10000 --batch-size 64 --eval-every 500 --seed 0'
!python -m gram.train {args}

## Go / no-go
Suppression results only mean something if the model actually learned Python and SQL **through its auxiliaries**.
Look for: full-profile Python/SQL loss far below the untrained ~9 nats, a clear rise when the module for that
domain is disabled, and little change in core loss.

In [ ]:
import json
rows = [json.loads(l) for l in open(f'{OUT}/eval.jsonl')]
r = rows[-1]; v = r['val']
print(f"step {r['step']}  tokens {r['tokens']}  {r['elapsed_s']/60:.1f} min")
print(f"{'profile':8s}" + ''.join(f'{d:>9s}' for d in ('core', 'python', 'sql')))
for p, row in v.items():
    print(f'{p:8s}' + ''.join(f'{row[d]:9.3f}' for d in ('core', 'python', 'sql')))
print('ΔL python when Python module off:', round(v['sql']['python'] - v['full']['python'], 3))
print('ΔL sql    when SQL module off:   ', round(v['python']['sql'] - v['full']['sql'], 3))
print('ΔL core   with both off:          ', round(v['core']['core'] - v['full']['core'], 3))